# Generative AI Mini-Project: Single Image Super-Resolution (x4) Challenge
**Master 2 — Specialty I2A**

## Overview and Objectives
Welcome to the Single Image Super-Resolution (SISR) Challenge! In this mini-project, your objective is to design, train, and improve a deep learning model capable of upscaling Low-Resolution (LR) images of size $64 \times 64$ into High-Resolution (HR) images of size $256 \times 256$ (upscaling factor $\times 4$).

This starter notebook provides a complete, working pipeline with a **basic baseline Convolutional Autoencoder**. Your task as a team (4 to 5 students maximum) is to modify and improve this baseline to achieve the highest possible scores on the unseen test set.

### Evaluation Metrics
Your generated test predictions (`submission.mat`) will be evaluated on the online challenge platform using the `sewar` library across three full-reference metrics:
1. **PSNR (Peak Signal-to-Noise Ratio):** Measured in dB (*Higher is better*).
2. **SSIM (Structural Similarity Index):** Ranges from 0 to 1 (*Higher is better*).
3. **RMSE (Root Mean Squared Error):** Pixel-level reconstruction error (*Lower is better*).

### Our approach at a glance
| | Baseline | Ours |
|---|---|---|
| Architecture | Convolutional autoencoder: 2-conv encoder, 2 `ConvTranspose2d` decoder (≈60k params) | Convolutional autoencoder with a **32×32 bottleneck**: residual blocks with **channel attention** in both encoder and decoder, a **U-Net skip connection**, **PixelShuffle** upsampling in the decoder (≈5.2M params) |
| What the net predicts | The whole HR image (through a `Sigmoid`) | Only the **residual** on top of a bicubic upscale |
| Rules | | No transformer, GAN, diffusion or pre-trained weights. Trained from scratch on `train_data` only. |
| Loss | MSE | **L1** (sharper results, better PSNR in practice, cf. EDSR) |
| Training data | Full 64×64 images, no augmentation | Random **48×48 LR crops** + the 8 **flip/rotation** symmetries |
| Optimisation | Adam 1e-3, 10 epochs | Adam 2e-4, linear warm-up + **cosine decay**, 25k iterations, fp16 mixed precision |
| Inference | Single forward pass | **×8 self-ensemble** (average over flips/rotations) |

## 1. Environment Setup and Dependencies

In [ ]:
# Install only the packages that are missing (Colab/Kaggle already ship torch, scipy, matplotlib and pillow).
import importlib.util, subprocess, sys

REQUIRED = {"sewar": "sewar", "scipy": "scipy", "torch": "torch", "matplotlib": "matplotlib", "PIL": "pillow"}
missing = [pkg for module, pkg in REQUIRED.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("Installed:", missing if missing else "nothing, all packages already present")

In [ ]:
import os
import glob
import math
import time
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from scipy.io import savemat

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, random_split

from sewar.full_ref import psnr, ssim, rmse

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True  # input sizes are fixed, let cuDNN pick the fastest kernels

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Computation Device: {device}" + (f" ({torch.cuda.get_device_name(0)})" if device.type == "cuda" else ""))

## 2. Dataset Extraction and DataLoader Preparation

Make sure you have uploaded and unzipped `train_data` and `test_lr` to your working directory.
- `train_data` contains paired `LR/` ($64 \times 64$) and `HR/` ($256 \times 256$) PNG patches.
- `test_lr` contains 20 unseen Low-Resolution test images (`001.png` to `020.png`) of size $64 \times 64$.

In [10]:
class SuperResolutionDataset(Dataset):
    """
    Custom PyTorch Dataset for loading paired LR (64x64) and HR (256x256) PNG images.
    Automatically searches inside root_dir for 'LR' and 'HR' folders.
    """
    def __init__(self, root_dir):
        lr_matches = glob.glob(os.path.join(root_dir, "**", "LR", "*.png"), recursive=True)
        hr_matches = glob.glob(os.path.join(root_dir, "**", "HR", "*.png"), recursive=True)
        
        self.lr_files = sorted(lr_matches)
        self.hr_files = sorted(hr_matches)
        
        if len(self.lr_files) == 0 or len(self.lr_files) != len(self.hr_files):
            raise RuntimeError(
                f"Dataset error: Found {len(self.lr_files)} LR images and {len(self.hr_files)} HR images in '{root_dir}'. "
                "Please verify that 'train_data.zip' is uploaded and extracted."
            )

    def __len__(self):
        return len(self.lr_files)

    def __getitem__(self, idx):
        # Load RGB images and normalize pixel values to [0.0, 1.0]
        lr_img = np.array(Image.open(self.lr_files[idx]).convert("RGB"), dtype=np.float32) / 255.0
        hr_img = np.array(Image.open(self.hr_files[idx]).convert("RGB"), dtype=np.float32) / 255.0

        # Convert from H x W x C (NumPy) to C x H x W (PyTorch Tensor)
        lr_tensor = torch.from_numpy(lr_img.transpose(2, 0, 1))
        hr_tensor = torch.from_numpy(hr_img.transpose(2, 0, 1))
        return lr_tensor, hr_tensor

In [ ]:
# Same 90% / 10% split as the baseline (same seed), so validation scores are directly comparable.
full_dataset = SuperResolutionDataset("train_data")
val_size = int(0.10 * len(full_dataset))
train_size = len(full_dataset) - val_size

train_dataset, val_dataset = random_split(
    full_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED)
)
print(f"Total Pairs: {len(full_dataset)} | Training Samples: {len(train_dataset)} | Validation Samples: {len(val_dataset)}")

### Preloading and data augmentation
The whole dataset is about 1 GB as float32, so we load it **once onto the GPU** instead of decoding PNGs every step.

Each training batch is built on the fly:
- **Random crops:** a random 48×48 window of a 64×64 LR image, paired with the *aligned* 192×192 HR window. LR pixel $(y, x)$ comes from HR pixels $(4y \dots 4y{+}3,\ 4x \dots 4x{+}3)$, so the HR offset is just $4\times$ the LR offset. Crops give translation diversity and make each step cheaper.
- **Dihedral augmentation:** each crop is transformed by one of the 8 symmetries of a square (4 rotations × optional horizontal flip). Bicubic downsampling commutes with these transforms, so every augmented pair is still a valid LR/HR pair. This multiplies the effective dataset size by 8.

In [ ]:
def stack_subset(subset):
    lrs, hrs = zip(*(subset[i] for i in range(len(subset))))
    return torch.stack(lrs).to(device), torch.stack(hrs).to(device)

train_lr, train_hr = stack_subset(train_dataset)
val_lr, val_hr = stack_subset(val_dataset)

SCALE = 4
LR_PATCH = 48
BATCH_SIZE = 16


def dihedral(x, k):
    "Apply symmetry k in [0, 8) of the square: optional horizontal flip (k >= 4), then k % 4 quarter-turns."
    if k >= 4:
        x = x.flip(-1)
    return torch.rot90(x, k % 4, dims=(-2, -1))


def inverse_dihedral(x, k):
    "Undo dihedral(x, k)."
    x = torch.rot90(x, -(k % 4), dims=(-2, -1))
    return x.flip(-1) if k >= 4 else x


def random_batch(batch_size=BATCH_SIZE, patch=LR_PATCH):
    "Random aligned LR/HR crops, each with a random flip/rotation."
    lr_out, hr_out = [], []
    max_offset = train_lr.shape[-1] - patch
    for i in torch.randint(0, len(train_lr), (batch_size,)).tolist():
        y, x = random.randint(0, max_offset), random.randint(0, max_offset)
        lr = train_lr[i, :, y:y + patch, x:x + patch]
        hr = train_hr[i, :, SCALE * y:SCALE * (y + patch), SCALE * x:SCALE * (x + patch)]
        k = random.randrange(8)
        lr_out.append(dihedral(lr, k))
        hr_out.append(dihedral(hr, k))
    return torch.stack(lr_out), torch.stack(hr_out)


lr_b, hr_b = random_batch()
print(f"train LR {tuple(train_lr.shape)}  train HR {tuple(train_hr.shape)}  val LR {tuple(val_lr.shape)}")
print(f"one training batch: LR {tuple(lr_b.shape)} -> HR {tuple(hr_b.shape)}")
assert all(torch.equal(inverse_dihedral(dihedral(hr_b, k), k), hr_b) for k in range(8))

## 3. Model: a residual convolutional autoencoder with skip connections
The challenge only allows **convolutional autoencoders**, so the model keeps the baseline's `encoder → latent → decoder` structure, made much stronger:

```
LR 64×64×3 ──► ENCODER ─────────────────────────────► latent 32×32×128 ──► DECODER ───────────────────────────────────────► + ──► SR 256×256×3
               stem conv → 4 RCAB (64×64×64)               │               6 RCAB (32×32) → ×2 PixelShuffle (64×64×64)        ▲
               → strided conv ↓2 → 6 RCAB (32×32×128) ─────┘               → concat skip → 1×1 conv → 4 RCAB (64×64×64)       │
                        │                                                   → ×2 → ×2 PixelShuffle → conv (256×256×3)         │
                        └──────────────── skip (64×64×64) ───────────────────────►┘                                            │
LR ─────────────────────────────────────────────── bicubic ×4 ────────────────────────────────────────────────────────────────┘
```

The baseline has three weaknesses: very little capacity, `ConvTranspose2d` upsampling (prone to checkerboard artifacts), and it must regenerate the *entire* image through a `Sigmoid`. We fix each of them:

1. **A real bottleneck.** The encoder compresses the image with a strided convolution to a 32×32 latent, so the deepest features have twice the receptive field for the same cost. The decoder reconstructs from that latent.
2. **Encoder–decoder skip connection (U-Net style).** Downsampling throws away fine spatial detail that SR needs. The 64×64 encoder features are therefore concatenated into the decoder at the same resolution.
3. **Residual blocks with channel attention, without BatchNorm.** Every stage is a group of residual blocks with an outer skip connection. Inside each block, a squeeze-and-excitation gate (global average pool → small MLP → sigmoid) rescales the feature channels. This is convolutional channel attention, not transformer self-attention. BatchNorm is allowed but left out on purpose: it normalises away the absolute intensity and contrast information that SR must reproduce, and EDSR showed that removing it improves quality and saves memory.
4. **Sub-pixel convolution (`PixelShuffle`) in the decoder.** Each ×2 step is a conv producing $4C$ channels, which are rearranged into a 2× larger $C$-channel map. This avoids the uneven-overlap checkerboard pattern of strided transposed convolutions.
5. **Bicubic residual learning.** The output is `bicubic(LR) + decoder(...)`. Bicubic already gets the low frequencies right (≈24.5 dB), so the decoder only has to reconstruct the missing **high-frequency detail**. That target is small, zero-mean and much easier to learn. The decoder's last conv is **zero-initialised**, so the untrained model *is* the bicubic baseline. (Set `bicubic_skip=False` to make the decoder produce the full image.)

The output is not passed through a sigmoid. It is clamped to $[0, 1]$ at inference time instead, so gradients do not vanish near black or white pixels.

In [ ]:
class ChannelAttention(nn.Module):
    "Squeeze-and-excitation: rescale each channel by a learned, image-dependent factor in (0, 1)."
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.gate = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, channels // reduction, 1), nn.ReLU(inplace=True),
            nn.Conv2d(channels // reduction, channels, 1), nn.Sigmoid(),
        )

    def forward(self, x):
        return x * self.gate(x)


class RCAB(nn.Module):
    "Residual Channel Attention Block: x + CA(conv(relu(conv(x)))). No BatchNorm."
    def __init__(self, channels):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1),
            ChannelAttention(channels),
        )

    def forward(self, x):
        return x + self.body(x)


class BicubicUpsampler(nn.Module):
    "Parameter-free reference (also used as the bicubic residual path of the autoencoder)."
    def __init__(self, scale=SCALE):
        super().__init__()
        self.scale = scale

    def forward(self, x):
        return F.interpolate(x, scale_factor=self.scale, mode="bicubic", align_corners=False)


class ResidualGroup(nn.Module):
    "n RCABs + a conv, wrapped in an outer skip connection."
    def __init__(self, channels, n_blocks):
        super().__init__()
        self.body = nn.Sequential(*[RCAB(channels) for _ in range(n_blocks)],
                                  nn.Conv2d(channels, channels, 3, padding=1))

    def forward(self, x):
        return x + self.body(x)


def subpixel_up(in_channels, out_channels):
    "x2 upsampling: conv to 4*out channels, then PixelShuffle rearranges them into a 2x larger map."
    return nn.Sequential(nn.Conv2d(in_channels, 4 * out_channels, 3, padding=1), nn.PixelShuffle(2))


class Encoder(nn.Module):
    "LR image (3, H, W) -> latent (2C, H/2, W/2), plus the (C, H, W) features for the skip connection."
    def __init__(self, c=64, n_blocks=4, n_latent_blocks=6):
        super().__init__()
        self.stem = nn.Conv2d(3, c, 3, padding=1)
        self.level1 = ResidualGroup(c, n_blocks)                   # 64x64, C channels
        self.down = nn.Conv2d(c, 2 * c, 3, stride=2, padding=1)    # 64x64 -> 32x32, 2C channels
        self.level2 = ResidualGroup(2 * c, n_latent_blocks)        # 32x32 bottleneck

    def forward(self, x):
        skip = self.level1(self.stem(x))
        latent = self.level2(self.down(skip))
        return latent, skip


class Decoder(nn.Module):
    "Latent (2C, H/2, W/2) + skip (C, H, W) -> image residual (3, 4H, 4W)."
    def __init__(self, c=64, n_blocks=4, n_latent_blocks=6, scale=SCALE):
        super().__init__()
        self.level2 = ResidualGroup(2 * c, n_latent_blocks)        # 32x32
        self.up_to_lr = subpixel_up(2 * c, c)                      # 32x32 -> 64x64
        self.fuse = nn.Conv2d(2 * c, c, 1)                         # merge [decoder, encoder skip]
        self.level1 = ResidualGroup(c, n_blocks)                   # 64x64
        self.up_to_hr = nn.Sequential(*[subpixel_up(c, c) for _ in range(int(math.log2(scale)))])  # 64 -> 128 -> 256
        self.tail = nn.Conv2d(c, 3, 3, padding=1)
        # Zero-init the last layer: at step 0 the decoder outputs 0, so the model starts as plain bicubic.
        nn.init.zeros_(self.tail.weight)
        nn.init.zeros_(self.tail.bias)

    def forward(self, latent, skip):
        x = self.up_to_lr(self.level2(latent))
        x = self.level1(self.fuse(torch.cat([x, skip], dim=1)))
        return self.tail(self.up_to_hr(x))


class SRAutoencoder(nn.Module):
    def __init__(self, c=64, n_blocks=4, n_latent_blocks=6, scale=SCALE, bicubic_skip=True):
        super().__init__()
        self.encoder = Encoder(c, n_blocks, n_latent_blocks)
        self.decoder = Decoder(c, n_blocks, n_latent_blocks, scale)
        self.bicubic = BicubicUpsampler(scale) if bicubic_skip else None

    def forward(self, x):
        latent, skip = self.encoder(x)
        out = self.decoder(latent, skip)
        return out + self.bicubic(x) if self.bicubic is not None else out


model = SRAutoencoder().to(device)
count = lambda m: sum(p.numel() for p in m.parameters() if p.requires_grad)
print(f"Encoder: {count(model.encoder):,} params | Decoder: {count(model.decoder):,} params")
print(f"Total Trainable Parameters: {count(model):,}")
with torch.no_grad():
    latent, skip = model.encoder(val_lr[:1])
    print(f"input {tuple(val_lr[:1].shape)} -> latent {tuple(latent.shape)} (+ skip {tuple(skip.shape)}) "
          f"-> output {tuple(model(val_lr[:1]).shape)}")

## 4. Training and loss curves
- **L1 loss** instead of MSE. MSE squares the error, so it is dominated by a few large errors on edges and settles for blurry averages elsewhere. L1 penalises all errors linearly and gives sharper reconstructions. EDSR/RCAN report higher PSNR with L1 even though PSNR is defined through MSE.
- **Adam** ($\beta_2 = 0.99$) with a short **linear warm-up** (500 iterations), then **cosine decay** from $2\times10^{-4}$ to $10^{-6}$. The low final learning rate is where most of the last fraction of a dB comes from.
- **Mixed precision** (fp16 autocast + `GradScaler`) roughly halves memory use and speeds up the convolutions on tensor cores.
- Every 500 iterations we measure L1 loss and PSNR on the full 120 validation images and keep the checkpoint with the best validation PSNR.

Training takes about 35 minutes on an RTX 3050 laptop GPU. If `sr_autoencoder_best.pth` already exists, the cell loads it (with its recorded curves) instead of retraining. Set `RETRAIN = True` to force a fresh run.

In [ ]:
TOTAL_ITERS = 25_000    # ~370 epoch-equivalents (68 batches of 16 per pass over the 1080 training images)
EVAL_EVERY = 500
PEAK_LR, MIN_LR = 2e-4, 1e-6
WARMUP_ITERS = 500
CKPT_PATH = "sr_autoencoder_best.pth"
RETRAIN = not os.path.exists(CKPT_PATH)
use_amp = device.type == "cuda"


def lr_at(it):
    "Linear warm-up, then cosine decay from PEAK_LR to MIN_LR."
    if it < WARMUP_ITERS:
        return PEAK_LR * (it + 1) / WARMUP_ITERS
    t = (it - WARMUP_ITERS) / max(1, TOTAL_ITERS - WARMUP_ITERS)
    return MIN_LR + 0.5 * (PEAK_LR - MIN_LR) * (1 + math.cos(math.pi * t))


@torch.no_grad()
def evaluate(net, lr_imgs, hr_imgs, batch_size=20):
    "(mean L1, mean PSNR in dB) on full images. PSNR uses uint8-rounded outputs, like sewar on the leaderboard."
    net.eval()
    total_l1, psnrs = 0.0, []
    for i in range(0, len(lr_imgs), batch_size):
        pred = net(lr_imgs[i:i + batch_size]).clamp(0, 1)
        hr = hr_imgs[i:i + batch_size]
        total_l1 += F.l1_loss(pred, hr, reduction="none").mean(dim=(1, 2, 3)).sum().item()
        mse = ((pred * 255).round() - (hr * 255).round()).pow(2).mean(dim=(1, 2, 3))
        psnrs += (10 * torch.log10(255.0 ** 2 / mse)).tolist()
    net.train()
    return total_l1 / len(lr_imgs), float(np.mean(psnrs))


bicubic_val_loss, bicubic_val_psnr = evaluate(BicubicUpsampler(), val_lr, val_hr)
print(f"Reference (torch bicubic) on validation: L1 {bicubic_val_loss:.5f} | PSNR {bicubic_val_psnr:.3f} dB\n")

criterion = nn.L1Loss()

if RETRAIN:
    optimizer = optim.Adam(model.parameters(), lr=PEAK_LR, betas=(0.9, 0.99))
    scaler = torch.amp.GradScaler(enabled=use_amp)
    history = {"iter": [], "lr": [], "train_loss": [], "val_loss": [], "val_psnr": []}
    best_psnr, best_state = -float("inf"), None
    running_loss, t0 = torch.zeros((), device=device), time.time()

    model.train()
    for it in range(TOTAL_ITERS):
        for group in optimizer.param_groups:
            group["lr"] = lr_at(it)

        lr_batch, hr_batch = random_batch()
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            preds = model(lr_batch)
        loss = criterion(preds.float(), hr_batch)

        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.detach()

        if (it + 1) % EVAL_EVERY == 0 or it + 1 == TOTAL_ITERS:
            n = EVAL_EVERY if (it + 1) % EVAL_EVERY == 0 else (it + 1) % EVAL_EVERY
            train_loss = running_loss.item() / n
            running_loss.zero_()
            val_loss, val_psnr = evaluate(model, val_lr, val_hr)
            for key, value in zip(history, (it + 1, lr_at(it), train_loss, val_loss, val_psnr)):
                history[key].append(value)

            marker = ""
            if val_psnr > best_psnr:
                best_psnr, marker = val_psnr, "  * best"
                best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
                torch.save({"model": best_state, "history": history, "best_psnr": best_psnr}, CKPT_PATH)
            print(f"iter {it + 1:6d}/{TOTAL_ITERS} | lr {lr_at(it):.2e} | train L1 {train_loss:.5f} | "
                  f"val L1 {val_loss:.5f} | val PSNR {val_psnr:.3f} dB | {(time.time() - t0) / 60:5.1f} min{marker}")

    # Final save so the checkpoint carries the complete curves, then keep the best weights.
    torch.save({"model": best_state, "history": history, "best_psnr": best_psnr}, CKPT_PATH)
    model.load_state_dict(best_state)
    print(f"\nTraining done in {(time.time() - t0) / 60:.1f} min. Best val PSNR: {best_psnr:.3f} dB -> saved to {CKPT_PATH}")
else:
    ckpt = torch.load(CKPT_PATH, map_location=device)
    model.load_state_dict(ckpt["model"])
    history, best_psnr = ckpt["history"], ckpt["best_psnr"]
    print(f"Loaded {CKPT_PATH} (best val PSNR {best_psnr:.3f} dB, {history['iter'][-1]:,} iterations). Set RETRAIN = True to retrain.")

# --- Training & validation curves ---
iters_k = np.array(history["iter"]) / 1000
fig, (ax_loss, ax_psnr) = plt.subplots(1, 2, figsize=(13, 4.5))

ax_loss.plot(iters_k, history["train_loss"], linewidth=2, label="Training loss (48x48 crops)")
ax_loss.plot(iters_k, history["val_loss"], linewidth=2, label="Validation loss (full images)")
ax_loss.axhline(bicubic_val_loss, color="gray", linestyle="--", label="Bicubic (validation)")
ax_loss.set(title="Training and Validation Loss Curve", xlabel="Iteration (x1000)", ylabel="Loss (L1)")
ax_loss.grid(True, linestyle="--", alpha=0.6)
ax_loss.legend()

ax_psnr.plot(iters_k, history["val_psnr"], linewidth=2, color="tab:green", label="Autoencoder (validation)")
ax_psnr.axhline(bicubic_val_psnr, color="gray", linestyle="--", label="Bicubic (validation)")
ax_psnr.set(title="Validation PSNR", xlabel="Iteration (x1000)", ylabel="PSNR (dB)")
ax_psnr.grid(True, linestyle="--", alpha=0.6)
ax_psnr.legend()

plt.tight_layout()
plt.show()

## 5. Local validation evaluation (`sewar`) and visual inspection
We compute the three official challenge metrics (**PSNR**, **SSIM** and **RMSE**) with `sewar.full_ref` on **all 120 validation images** for three methods:
- **Bicubic (PIL)**, the classical interpolation reference.
- **Autoencoder**, a single forward pass.
- **Autoencoder + ×8 self-ensemble.** At test time we run the model on all 8 flipped/rotated versions of the input, undo each transform on the output, and average. The errors in the 8 predictions are partly independent, so averaging them gives a free gain (about +0.1 dB in EDSR's experiments) at 8× inference cost, which is negligible for 20 test images.

In [ ]:
def tensor_to_uint8_hwc(tensor):
    "(C, H, W) float tensor in [0, 1] -> (H, W, C) uint8 array in [0, 255]."
    img_np = tensor.detach().cpu().numpy().transpose(1, 2, 0)
    return np.clip(img_np * 255.0, 0, 255).round().astype(np.uint8)


@torch.no_grad()
def super_resolve(net, lr_imgs, self_ensemble=True, chunk=10):
    "(N, 3, 64, 64) -> (N, 3, 256, 256) in [0, 1], optionally averaged over the 8 dihedral transforms."
    net.eval()
    outs = []
    for lr_batch in lr_imgs.split(chunk):  # small chunks keep GPU memory low
        if self_ensemble:
            out = sum(inverse_dihedral(net(dihedral(lr_batch, k)), k) for k in range(8)) / 8
        else:
            out = net(lr_batch)
        outs.append(out.clamp(0, 1))
    return torch.cat(outs)


def bicubic_pil(lr_t):
    lr_img = Image.fromarray(tensor_to_uint8_hwc(lr_t))
    return np.array(lr_img.resize((256, 256), Image.BICUBIC))


def score(gt, pred):
    return float(psnr(gt, pred)), float(ssim(gt, pred)[0]), float(rmse(gt, pred))  # ssim returns (ssim, cs)


gt_val = [tensor_to_uint8_hwc(t) for t in val_hr]
pred_single = [tensor_to_uint8_hwc(t) for t in super_resolve(model, val_lr, self_ensemble=False)]
pred_ensemble = [tensor_to_uint8_hwc(t) for t in super_resolve(model, val_lr, self_ensemble=True)]
pred_bicubic = [bicubic_pil(t) for t in val_lr]

results = {}
for name, preds in [("Bicubic (PIL)", pred_bicubic), ("Autoencoder", pred_single), ("Autoencoder + x8 self-ensemble", pred_ensemble)]:
    results[name] = np.mean([score(g, p) for g, p in zip(gt_val, preds)], axis=0)

print("=" * 72)
print(f"   VALIDATION SCORES (sewar, all {len(gt_val)} validation images)")
print("=" * 72)
print(f"{'Method':<34}{'PSNR (dB) ↑':>13}{'SSIM ↑':>11}{'RMSE ↓':>11}")
print("-" * 72)
for name, (p, s, r) in results.items():
    print(f"{name:<34}{p:>13.4f}{s:>11.4f}{r:>11.4f}")
print("=" * 72)
gain = results["Autoencoder + x8 self-ensemble"][0] - results["Bicubic (PIL)"][0]
print(f"Gain over bicubic: {gain:+.3f} dB")

In [ ]:
# --- Visual comparison on 3 validation samples, with a 2x zoom on the centre region ---
def zoom(img, size=96):
    h, w = img.shape[:2]
    y, x = (h - size) // 2, (w - size) // 2
    return img[y:y + size, x:x + size]

columns = [
    ("LR input (64x64)", lambda i: np.array(Image.fromarray(tensor_to_uint8_hwc(val_lr[i])).resize((256, 256), Image.NEAREST))),
    ("Bicubic", lambda i: pred_bicubic[i]),
    ("Autoencoder + self-ensemble", lambda i: pred_ensemble[i]),
    ("Ground truth HR (256x256)", lambda i: gt_val[i]),
]
fig, axes = plt.subplots(6, 4, figsize=(14, 21))
for row in range(3):
    for col, (title, get) in enumerate(columns):
        img = get(row)
        label = title if col in (0, 3) else f"{title} ({psnr(gt_val[row], img):.2f} dB)"
        axes[2 * row, col].imshow(img)
        axes[2 * row, col].set_title(f"Sample {row + 1}: {label}", fontsize=10)
        axes[2 * row + 1, col].imshow(zoom(img), interpolation="nearest")
        axes[2 * row + 1, col].set_title("centre crop, zoomed", fontsize=9)
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 6. Generate Test Set Predictions (`submission.mat`)
**Important Submission Instructions:**
- The cell below reads the 20 unseen Low-Resolution PNG images (`001.png` to `020.png`) from `test_lr/`.
- It generates the 20 reconstructed High-Resolution images ($256 \times 256 \times 3$, `uint8`) and packages them into a single compressed MATLAB file named **`submission.mat`** under the key `"predictions"` with exact shape `(20, 256, 256, 3)`.
- Upload **`submission.mat`** directly to the PythonAnywhere challenge website to view your live scores on the leaderboard.

In [ ]:
def find_test_lr_image(index, root_folder="test_lr"):
    "Locates 001.png .. 020.png inside test_lr/."
    fname = f"{index:03d}.png"
    direct_path = os.path.join(root_folder, fname)
    if os.path.exists(direct_path):
        return direct_path
    matches = glob.glob(os.path.join(root_folder, "**", fname), recursive=True)
    if matches:
        return matches[0]
    raise FileNotFoundError(f"Could not find test image '{fname}' inside '{root_folder}/'.")


test_lr = []
for i in range(1, 21):
    # Load 64x64 LR PNG image and normalize to [0.0, 1.0]
    lr_img = np.array(Image.open(find_test_lr_image(i, "test_lr")).convert("RGB"), dtype=np.float32) / 255.0
    test_lr.append(torch.from_numpy(lr_img.transpose(2, 0, 1)))
test_lr = torch.stack(test_lr).to(device)

# Generate the 256x256 HR predictions with the x8 self-ensemble
predictions_array = np.stack([tensor_to_uint8_hwc(t) for t in super_resolve(model, test_lr, self_ensemble=True)], axis=0)

# Strict verification before saving
assert predictions_array.shape == (20, 256, 256, 3), f"Invalid shape: {predictions_array.shape}. Expected (20, 256, 256, 3)."
assert predictions_array.dtype == np.uint8, f"Invalid dtype: {predictions_array.dtype}. Expected uint8."

# Save only the evaluation file as submission.mat
savemat("submission.mat", {"predictions": predictions_array}, do_compression=True)
size_mb = os.path.getsize("submission.mat") / (1024 * 1024)

print(f"SUCCESS! Generated 'submission.mat' ({size_mb:.2f} MB)")
print(f"Array key: 'predictions' | Shape: {predictions_array.shape} | Dtype: {predictions_array.dtype}")
print("You can now upload 'submission.mat' to the evaluation website!")